# Evaluation of distribution of models on CORDEX-ML_BENCH datasets

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import functools
import math
import string

import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis import plot_map, SUBREGIONS, BOX_LOCATIONS
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import mean_bias, std_bias, stat_bias, plot_freq_density, plot_distribution_figure, compute_metrics, DIST_THRESHOLDS, plot_freq_density_figure
from mlde_analysis.wet_dry import threshold_exceeded_prop_stats, threshold_exceeded_prop, threshold_exceeded_prop_error, threshold_exceeded_prop_change, plot_threshold_exceedence_errors, THRESHOLDS, wd_mean, wd_mean_bias
from mlde_utils import cp_model_rotated_pole
from mlde_analysis import qq_plot, reasonable_quantiles

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS["CPM"]

## Figure: distribution

* Frequency Density Histogram of rainfall intensities
* Maps of Mean bias ($\frac{\mu_{sample}-\mu_{CPM}}{\mu_{CPM}}$) over all samples, time and ensemble members
* Std Dev Bias $\frac{\sigma_{sample}}{\sigma_{CPM}}$ over all samples, time and ensemble members

Table of:

* RMS biases
* J-S Distances
* proportion of density over thresholds

In [ ]:
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    
    hist_das = PRED_DAS[var]
    target_da = TARGET_DAS[var]
    
    metrics_ds = compute_metrics(hist_das, target_da, thresholds=DIST_THRESHOLDS[var])

    pretty_table(metrics_ds, round=4)
    
    normalize=(var == "pr")
    mean_biases = PRED_DAS[var].groupby("model").map(mean_bias, target_da=target_da, normalize=(var=="pr"))

    std_biases = PRED_DAS[var].groupby("model").map(std_bias, target_da=target_da, normalize=(var=="pr"))
    f_q999_bias = functools.partial(stat_bias, stat_func=functools.partial(xr.DataArray.quantile, q=0.999))
    q999_biases = PRED_DAS[var].groupby("model").map(f_q999_bias, target_da=target_da, normalize=(var=="pr"))
    
    bias_kwargs = {"style": f"{var}Bias"}
    for fd_kwargs in [{"yscale": "log", "target_label": target_sim_key}]:
        fig = plt.figure(layout="constrained", figsize=(5.5, 7))
        error_fig = plt.figure(layout="constrained", figsize=(5.5, 2.5))
        error_axd = error_fig.subplot_mosaic([["Error"]])
        error_ax = error_axd["Error"]
        axd = plot_distribution_figure(
            fig,
            hist_das,
            target_da,
            {"meanb": mean_biases, "stdb": std_biases, "q999b": q999_biases,},
            MODELLABEL2SPEC,
            error_ax=error_ax, hrange=VAR_RANGES[var], fd_kwargs=fd_kwargs, bias_kwargs=bias_kwargs
        )
        if var == "relhum150cm":
            axd["Density"].axvline(x=100, color='k', linestyle='--', linewidth=1)
        
        plt.show()

## QQ plots

In [ ]:
quantile_dims=["ensemble_member", "T", "X", "Y"]

for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)

    target_da = TARGET_DAS[var]
    quantiles = reasonable_quantiles(target_da)
    target_quantiles = target_da.cf.quantile(quantiles, dim=quantile_dims).rename("target_q")

    for source, ds in EVAL_DS.items():
        pred_da = ds[f"pred_{var}"]
        pred_quantiles = pred_da.cf.quantile(quantiles, dim=quantile_dims).rename("pred_q")

        layout="constrained"

        fig, ax = plt.subplots(figsize=(3.5, 3.5), layout="constrained")

        xlabel = f"CPM \n{xr.plot.utils.label_from_attrs(da=target_da)}"
        ylabel = f"Predicted \n{xr.plot.utils.label_from_attrs(da=pred_da)}"

        qq_plot(ax, target_quantiles, pred_quantiles, title=f"Predicted quantiles vs Target quantiles", xlabel=xlabel, ylabel=ylabel)

    plt.show()

## Figure: Threshold exceedence

In [ ]:
for var, thresholds in THRESHOLDS.items():
    if var in eval_vars:

        target_da = TARGET_DAS[var]
        
        threshold_exceeded_stats = { threshold: threshold_exceeded_prop_stats(VAR_DAS[var][f"pred_{var}"], target_da, threshold) for threshold in thresholds }

        dfs = [threshold_exceeded_stats[threshold].cf.mean(["X", "Y"]).to_dataframe().style.set_table_attributes("style='display:inline'").set_caption(f"Threshold: {threshold}{target_da.attrs['units']}").format(precision=1).to_html() for threshold in thresholds]

        IPython.display.display_html(functools.reduce(lambda v, e: v+e, dfs), raw=True)

        for threshold in thresholds:
            IPython.display.display_markdown(f"#### Threshold: {threshold}{target_da.attrs['units']}", raw=True)

            plot_threshold_exceedence_errors(threshold_exceeded_stats[threshold])

            plt.show()

## Rough: wet-day mean